In [47]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o-mini")

# conn = sqlite3.connect("memory.db", check_same_thread=False)

# llm.invoke([{"role": "user", "content": "안녕?"}])

In [48]:


class State(MessagesState):
    custom_stuff: str

graph_builder = StateGraph(State)


In [49]:
@tool
def get_weather(city: str):
    """Gets weather in city """
    return f"The weather in {city} is sunny"

llm_with_tools = llm.bind_tools(tools=[get_weather])

def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {
        "messages": [response]
    }

In [50]:
tool_node = ToolNode(
    tools=[get_weather]
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    #checkpointer=SqliteSaver(conn)
)


In [ ]:
async for event in graph.astream(
    {
        "messages": [{"role": "user", "content": "서울과 인천과 부산의 날씨는 어때?"}],        
    },
    stream_mode="values"
    # config={
    #     "configurable": {
    #         "thread_id": "2",
    #     },
    # }
):
    print(event)


(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019d109f-f3c1-70a2-b9e7-169b90915ce6', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[]), {'langgraph_step': 1, 'langgraph_node': 'chatbot', 'langgraph_triggers': ('branch:to:chatbot',), 'langgraph_path': ('__pregel_pull', 'chatbot'), 'langgraph_checkpoint_ns': 'chatbot:69ebf461-fbe6-4fb8-c9be-fa57b72d047e', 'checkpoint_ns': 'chatbot:69ebf461-fbe6-4fb8-c9be-fa57b72d047e', 'ls_provider': 'openai', 'ls_model_name': 'gpt-4o-mini', 'ls_model_type': 'chat', 'ls_temperature': None})
(AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model_provider': 'openai'}, id='lc_run--019d109f-f3c1-70a2-b9e7-169b90915ce6', tool_calls=[{'name': 'get_weather', 'args': {}, 'id': 'call_HxMiCG8fBEGMW4FueCp8xQSR', 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=[{'name': 'get_weather', 'args': '', 'id': 'call_HxMiCG8fBEGMW4FueCp8xQSR', 'index': 0, 'type': 'too

In [52]:


for state in graph.get_state_history({
    "configurable": {
        "thread_id": "2",
    },
}):
    print(state.next)

ValueError: No checkpointer set